# LangChain: Models, Prompts and Output Parsers (Groq Llama 3.1)

## Outline
* Direct API calls to Groq
* API calls through LangChain:
  * Prompts
  * Models  
  * Output parsers

**Model Used:** Groq Llama-3.1-8b-instant

In [1]:
# Cell 2: Install Required Packages

# Install required packages
!pip install langchain langchain-groq python-dotenv -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 3: Import Libraries and Load Environment
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file in parent directory
load_dotenv(find_dotenv())

# Verify API key is loaded
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Groq API Key loaded successfully")

✅ Groq API Key loaded successfully


In [3]:
# Cell 4: Initialize Groq Model with LangChain
from langchain_groq import ChatGroq

# Initialize the Groq LLM
llm_model = "llama-3.1-8b-instant"

# Create chat model with temperature=0 for consistent outputs
chat = ChatGroq(
    temperature=0.0,
    model=llm_model,
    groq_api_key=groq_api_key
)

print(f"✅ Model initialized: {llm_model}")
print(chat)

d:\User\Desktop\Acads\4th Year\TwoTabs Dir\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Model initialized: llama-3.1-8b-instant
profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x000001F1942F4440> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F1942F4EC0> model_name='llama-3.1-8b-instant' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********')


In [5]:
# Cell 5: Test Basic Completion

# Test the model with a simple question
from langchain_core.messages import HumanMessage

test_message = [HumanMessage(content="What is 1+1?")]
response = chat.invoke(test_message)
print(response.content)

1 + 1 = 2.


In [ ]:
# Cell 6: Define Customer Email Example
# Customer email in "pirate English"
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

print("Customer Email:")
print(customer_email)

Customer Email:

Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!



In [ ]:
# Cell 7: Create Prompt Template for Style Translation
from langchain_core.prompts import ChatPromptTemplate

# Define the template string
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

# Create prompt template
prompt_template = ChatPromptTemplate.from_template(template_string)

print("✅ Prompt template created")
print(f"Input variables: {prompt_template.messages[0].prompt.input_variables}")

✅ Prompt template created
Input variables: ['style', 'text']


In [ ]:
# Cell 8: Translate Customer Email to Professional Style

# Define target style
customer_style = """American English \
in a calm and respectful tone
"""

# Format the prompt with actual values
customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)

print("Formatted Prompt:")
print(customer_messages[0].content)
print("\n" + "="*50 + "\n")

# Get response from LLM
customer_response = chat(customer_messages)

print("Translated Response:")
print(customer_response.content)


In [ ]:
# Cell 9: Reverse Translation - Professional to Pirate
# Service reply in professional English
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

# Target style: Pirate English
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

# Reuse the same prompt template
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply
)

print("Service Message (to be translated to Pirate):")
print(service_messages[0].content)
print("\n" + "="*50 + "\n")

# Get pirate translation
service_response = chat(service_messages)

print("Pirate Translation:")
print(service_response.content)


In [ ]:
# Cell 10: Output Parsing - Define Customer Review
## Output Parsers

Now let's explore structured output parsing. We'll extract specific information from a product review and format it as JSON.

**Desired Output Format:**
```json
{
  "gift": false,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}


In [ ]:
# Cell 11: Customer Review Text
customer_review = """\
This leaf blower is pretty amazing. It has four settings: \
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

print("Customer Review:")
print(customer_review)

In [ ]:
# Cell 12: First Attempt - Manual JSON Request
# Create a review template asking for JSON output
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price, \
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

# Create prompt and get response
prompt_template = ChatPromptTemplate.from_template(review_template)
messages = prompt_template.format_messages(text=customer_review)

response = chat(messages)
print("Raw Response:")
print(response.content)
print(f"\nResponse Type: {type(response.content)}")


In [ ]:
# Cell 13: Demonstrate the String Problem
# This will cause an error because response.content is a string, not a dict
try:
    response.content.get('gift')
except AttributeError as e:
    print(f"❌ Error: {e}")
    print(f"\nThe response is a {type(response.content)}, not a dictionary!")
    print("We need to parse it properly...")


In [ ]:
# Cell 14: Define Response Schemas

from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

# Define schemas for each field we want to extract
gift_schema = ResponseSchema(
    name="gift",
    description="Was the item purchased as a gift for someone else? \
    Answer True if yes, False if not or unknown."
)

delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="How many days did it take for the product to arrive? \
    If this information is not found, output -1."
)

price_value_schema = ResponseSchema(
    name="price_value",
    description="Extract any sentences about the value or price, \
    and output them as a comma separated Python list."
)

# Combine schemas
response_schemas = [
    gift_schema,
    delivery_days_schema,
    price_value_schema
]

print("✅ Response schemas defined")


In [ ]:
# Cell 15: Create Structured Output Parser

# Create the output parser
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Get format instructions
format_instructions = output_parser.get_format_instructions()

print("Format Instructions for LLM:")
print("="*60)
print(format_instructions)
